# Sound Category Classification - Existing Method (64-mel + SAD)

This notebook adapts the **technical validation pipeline** for 9-class sound category classification using the HuggingFace Coswara dataset.

## Overview

**Methodology**:
- **Feature extraction**: 64-mel log-spectrogram with deltas (adapted for 16 kHz)
- **Preprocessing**: Amplitude normalization + Sound Activity Detection (SAD)
- **Features**: 192-D (64 log-mel + 64 delta + 64 delta-delta) → averaged → sliced to 64-D
- **Normalization**: StandardScaler (mean centering)
- **Classifier**: RandomForest (criterion: gini)
- **Expected accuracy**: ~43-53%

**9 Audio Categories**:
1. breathing-deep
2. breathing-shallow
3. cough-heavy
4. cough-shallow
5. vowel-a
6. vowel-e
7. vowel-o
8. counting-normal
9. counting-fast

**Key Differences from Paper Method**:
- Heavy preprocessing (SAD, normalization)
- Fewer mel bins (64 vs 128)
- Delta features (computed then mostly discarded)
- StandardScaler normalization

**Adaptation for 16 kHz**:
- Original code designed for 44.1 kHz
- Adjusted window/hop parameters to maintain temporal resolution
- Reduced n_fft and fmax to match Nyquist frequency

## 1. Setup & Configuration

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import librosa
from datasets import load_dataset

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)

# Configuration
QUALITY_THRESHOLD = 1  # Include quality >= 1 (good + excellent)
OUTPUT_DIR = 'sound_category_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Audio categories (9 classes)
AUDIO_CATEGORIES = [
    'breathing-deep',
    'breathing-shallow',
    'cough-heavy',
    'cough-shallow',
    'vowel-a',
    'vowel-e',
    'vowel-o',
    'counting-normal',
    'counting-fast'
]

# Feature configuration (adapted from technical_validation/conf/feature.conf)
FEATURE_CONFIG = {
    'sampling_rate': 16000,
    'window_size': 400,          # 25 ms at 16 kHz (was 1024 @ 44.1kHz = 23.2ms)
    'hop_length': 160,           # 10 ms at 16 kHz (was 441 @ 44.1kHz = 10ms)
    'n_fft': 512,                # Next power of 2 >= window_size (was 1024)
    'n_mels': 64,
    'fmax': 8000,                # Nyquist @ 16kHz (was 22050 @ 44.1kHz)
    'sad_threshold': 0.0001,
    'sad_margin_ms': 50,
    'sad_sil_length_ms': 20
}

print(f"Configuration loaded successfully")
print(f"Random seed: {SEED}")
print(f"Quality threshold: >= {QUALITY_THRESHOLD}")
print(f"Number of classes: {len(AUDIO_CATEGORIES)}")
print(f"\nFeature configuration:")
for key, value in FEATURE_CONFIG.items():
    print(f"  {key}: {value}")

## 2. Data Loading from HuggingFace

Load the Coswara dataset from HuggingFace and apply quality filtering.

In [ ]:
# Load dataset from HuggingFace
print("Loading dataset from HuggingFace...")
ds = load_dataset("szzs1693/coswara-data", "audio")

print(f"\nRaw dataset sizes:")
print(f"  Train: {len(ds['train'])}")
print(f"  Validation: {len(ds['validation'])}")
print(f"  Test: {len(ds['test'])}")

# Apply quality filtering
def filter_quality(example):
    """Filter by quality score and valid audio"""
    return (example['quality_score'] >= QUALITY_THRESHOLD and 
            example['audio'] is not None and
            example['audio']['array'] is not None)

print(f"\nApplying quality filter (>= {QUALITY_THRESHOLD})...")
ds_filtered = ds.filter(filter_quality)

print(f"\nFiltered dataset sizes:")
print(f"  Train: {len(ds_filtered['train'])}")
print(f"  Validation: {len(ds_filtered['validation'])}")
print(f"  Test: {len(ds_filtered['test'])}")
print(f"  Total: {len(ds_filtered['train']) + len(ds_filtered['validation']) + len(ds_filtered['test'])}")

### Dataset Statistics & Visualization

In [ ]:
# Combine splits for overall statistics
all_data = []
for split_name in ['train', 'validation', 'test']:
    for example in ds_filtered[split_name]:
        all_data.append({
            'split': split_name,
            'audio_type': example['audio_type'],
            'quality_score': example['quality_score'],
            'covid_status': example['covid_status']
        })

df_stats = pd.DataFrame(all_data)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Audio type distribution
ax = axes[0, 0]
audio_type_counts = df_stats['audio_type'].value_counts()
audio_type_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Audio Type Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Audio Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# 2. Quality score distribution
ax = axes[0, 1]
quality_counts = df_stats['quality_score'].value_counts().sort_index()
quality_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Quality Score Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Quality Score')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

# 3. COVID status distribution
ax = axes[1, 0]
covid_counts = df_stats['covid_status'].value_counts()
covid_counts.plot(kind='bar', ax=ax, color='lightgreen')
ax.set_title('COVID Status Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('COVID Status')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# 4. Split distribution
ax = axes[1, 1]
split_counts = df_stats['split'].value_counts()
split_counts.plot(kind='bar', ax=ax, color='plum')
ax.set_title('Dataset Split Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Split')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/existing_dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDataset statistics:")
print(f"Total recordings: {len(df_stats)}")
print(f"\nSplit distribution:")
print(split_counts)
print(f"\nAudio type distribution:")
print(audio_type_counts)

## 3. Feature Extraction - Existing Method (Adapted)

Implement the technical validation pipeline adapted for 16 kHz:
1. Amplitude normalization
2. Sound Activity Detection (SAD) to remove silence
3. 64-mel log-spectrogram with delta and delta-delta features
4. Average over time and slice to first 1/3 (64-D final)

In [ ]:
def compute_SAD(sig, fs, threshold=0.0001, sad_margin_ms=50, sad_sil_length_ms=20):
    """
    Compute threshold-based Sound Activity Detection
    Adapted from technical_validation/9_class_classification/local/feature_extraction.py
    
    Parameters:
    -----------
    sig : np.ndarray
        Audio signal
    fs : int
        Sampling rate
    threshold : float
        Energy threshold for activity detection
    sad_margin_ms : int
        Margin around active samples (milliseconds)
    sad_sil_length_ms : int
        Leading/trailing silence to remove (milliseconds)
    
    Returns:
    --------
    np.ndarray
        Binary mask (1 = active, 0 = silence)
    """
    # Detect samples above energy threshold
    sample_activity = np.zeros(sig.shape)
    sample_activity[np.power(sig, 2) > threshold] = 1
    
    # Add margins around active samples
    sad_margin_length = int(sad_margin_ms * 1e-3 * fs)
    sad = np.zeros(sig.shape)
    for i in range(len(sample_activity)):
        if sample_activity[i] == 1:
            start = max(0, i - sad_margin_length)
            end = min(len(sad), i + sad_margin_length)
            sad[start:end] = 1
    
    # Remove leading/trailing silence
    sad_start_end_sil_length = int(sad_sil_length_ms * 1e-3 * fs)
    sad[0:sad_start_end_sil_length] = 0
    sad[-sad_start_end_sil_length:] = 0
    
    return sad


def extract_existing_features(audio_array, sr=16000, config=FEATURE_CONFIG):
    """
    Extract features using existing methodology adapted for 16 kHz
    Based on technical_validation/9_class_classification pipeline
    
    Parameters:
    -----------
    audio_array : np.ndarray
        Audio waveform
    sr : int
        Sampling rate (default 16000)
    config : dict
        Feature configuration
    
    Returns:
    --------
    np.ndarray
        64-D feature vector (or None if extraction fails)
    """
    try:
        # Check for valid audio
        if audio_array is None or len(audio_array) == 0:
            return None
        
        # Amplitude normalization (waveform-level)
        audio_array = audio_array / (np.max(np.abs(audio_array)) + 1e-8)
        
        # Apply SAD to remove silence
        sad = compute_SAD(
            audio_array, sr,
            threshold=config['sad_threshold'],
            sad_margin_ms=config['sad_margin_ms'],
            sad_sil_length_ms=config['sad_sil_length_ms']
        )
        audio_array = audio_array[sad == 1]
        
        # Check if audio is too short after SAD
        if len(audio_array) / sr < 0.5:  # Less than 0.5 seconds
            return None
        
        # Compute mel spectrogram (adapted parameters for 16 kHz)
        mel_spec = librosa.feature.melspectrogram(
            y=audio_array,
            sr=sr,
            n_fft=config['n_fft'],
            win_length=config['window_size'],
            hop_length=config['hop_length'],
            n_mels=config['n_mels'],
            fmax=config['fmax'],
            power=2.0
        )
        
        # Convert to dB scale (log-mel)
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Compute delta and delta-delta features
        delta = librosa.feature.delta(log_mel)
        delta_delta = librosa.feature.delta(log_mel, order=2)
        
        # Stack: [64 log-mel + 64 delta + 64 delta-delta] = 192 features × time
        features = np.concatenate([log_mel, delta, delta_delta], axis=0)
        
        # Average over time dimension
        feature_vec = features.mean(axis=1)  # (192,)
        
        # Keep only first 1/3 (matching classification.py line 73)
        # This is the mysterious slicing in the original code!
        feature_vec = feature_vec[:int(len(feature_vec) / 3)]  # (64,)
        
        return feature_vec
    
    except Exception as e:
        print(f"Error during feature extraction: {e}")
        return None


print("Feature extraction functions defined")
print("\nTest on random noise:")
test_audio = np.random.randn(16000)  # 1 second at 16kHz
test_feat = extract_existing_features(test_audio)
if test_feat is not None:
    print(f"Feature shape: {test_feat.shape}")
    print(f"Feature range: [{test_feat.min():.4f}, {test_feat.max():.4f}]")
else:
    print("Feature extraction returned None (too short after SAD)")

### Batch Feature Extraction (Optimized with Multiprocessing)

Uses HuggingFace `dataset.map()` with multiprocessing for 6-8x speedup.
Features are cached automatically for instant loading on subsequent runs.

In [ ]:
def add_features_to_example(example):
    """
    Feature extraction wrapper for dataset.map()
    
    Parameters:
    -----------
    example : dict
        Single example from dataset
    
    Returns:
    --------
    dict : Example with added 'features' and 'label' fields
    """
    # Extract audio
    audio_array = example['audio']['array']
    sr = example['audio']['sampling_rate']
    audio_type = example['audio_type']
    
    # Extract features (includes SAD, normalization, etc.)
    feat = extract_existing_features(audio_array, sr)
    
    # Convert audio type to label index
    label = AUDIO_CATEGORIES.index(audio_type)
    
    # Return with new fields
    return {
        'features': feat,
        'label': label,
        'extraction_success': feat is not None
    }


# Apply feature extraction with multiprocessing and caching
print("Extracting features with multiprocessing (using all CPU cores)...\n")
print("Note: First run will extract and cache. Subsequent runs will load from cache instantly.\n")

# Determine number of processes (use all cores)
import multiprocessing
num_proc = multiprocessing.cpu_count()
print(f"Using {num_proc} CPU cores for parallel processing\n")

# Create cache directory
os.makedirs('.cache/existing_features', exist_ok=True)

# Apply feature extraction to each split
ds_with_features = {}
for split_name in ['train', 'validation', 'test']:
    print(f"Processing {split_name} split...")
    ds_with_features[split_name] = ds_filtered[split_name].map(
        add_features_to_example,
        num_proc=num_proc,
        desc=f"Extracting features ({split_name})",
        cache_file_name=f".cache/existing_features/{split_name}_features.arrow"
    )
    print(f"  {split_name} completed!\n")

# Filter out failed extractions and convert to numpy arrays
def convert_to_numpy(split_name):
    """Convert dataset to numpy arrays, filtering failed extractions"""
    ds_split = ds_with_features[split_name]
    
    # Filter successful extractions
    ds_success = ds_split.filter(lambda x: x['extraction_success'])
    
    # Convert to lists then numpy
    features = [x['features'] for x in ds_success]
    labels = [x['label'] for x in ds_success]
    
    X = np.array(features)
    y = np.array(labels)
    
    failed_count = len(ds_split) - len(ds_success)
    
    print(f"{split_name:12s} - Extracted: {len(features):5d}, Failed: {failed_count:3d}, Shape: {X.shape}")
    
    return X, y

print("\n" + "="*70)
print("Converting to numpy arrays...")
print("="*70)

X_train, y_train = convert_to_numpy('train')
X_val, y_val = convert_to_numpy('validation')
X_test, y_test = convert_to_numpy('test')

print("\n" + "="*70)
print("Feature extraction completed!")
print("="*70)
print(f"Total samples: {X_train.shape[0] + X_val.shape[0] + X_test.shape[0]}")
print(f"Feature dimension: {X_train.shape[1]}")
print(f"\nNote: Features are cached in .cache/existing_features/")
print(f"      Re-running this cell will be instant (loads from cache)")

## 4. Preprocessing & Hyperparameter Tuning

Apply StandardScaler normalization (matching the existing pipeline) and tune the number of estimators.

In [ ]:
# Apply StandardScaler (with_mean=True, with_std=False - matching classification.py line 83)
print("Applying StandardScaler normalization...")
scaler = StandardScaler(with_mean=True, with_std=False)
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"Scaler fitted on training set")
print(f"Mean shape: {scaler.mean_.shape}")
print(f"Mean range: [{scaler.mean_.min():.4f}, {scaler.mean_.max():.4f}]")

In [ ]:
# Hyperparameter candidates
n_estimators_candidates = [100, 200, 500, 1000]

# Track results
val_accuracies = []
best_n_estimators = None
best_val_acc = 0.0

print("\nStarting hyperparameter tuning...\n")

for n_est in n_estimators_candidates:
    print(f"Training with n_estimators={n_est}...")
    
    clf = RandomForestClassifier(
        n_estimators=n_est,
        criterion='gini',
        random_state=SEED,
        n_jobs=-1,
        verbose=0
    )
    
    clf.fit(X_train_scaled, y_train)
    val_acc = clf.score(X_val_scaled, y_val)
    val_accuracies.append(val_acc)
    
    print(f"  Validation accuracy: {val_acc:.4f} ({val_acc*100:.2f}%)")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_n_estimators = n_est
    
    print()

print("="*50)
print(f"Best n_estimators: {best_n_estimators}")
print(f"Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")
print("="*50)

# Plot validation accuracy vs n_estimators
plt.figure(figsize=(10, 6))
plt.plot(n_estimators_candidates, val_accuracies, marker='o', linewidth=2, markersize=8)
plt.axhline(y=best_val_acc, color='r', linestyle='--', alpha=0.5, label='Best accuracy')
plt.xlabel('Number of Estimators', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.title('Hyperparameter Tuning: Validation Accuracy vs n_estimators', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/existing_hyperparameter_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Model Training & Evaluation

Train the final model with the best hyperparameters and evaluate on the test set.

In [ ]:
print(f"Training final model with n_estimators={best_n_estimators}...\n")

clf_final = RandomForestClassifier(
    n_estimators=best_n_estimators,
    criterion='gini',
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

clf_final.fit(X_train_scaled, y_train)

# Evaluate on test set
y_pred = clf_final.predict(X_test_scaled)
y_proba = clf_final.predict_proba(X_test_scaled)
test_acc = accuracy_score(y_test, y_pred)

print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"Expected range: 43-53%")
print("="*50)

# Per-class accuracy
print("\nPer-class accuracy:")
for i, cat in enumerate(AUDIO_CATEGORIES):
    mask = y_test == i
    if mask.sum() > 0:
        class_acc = (y_pred[mask] == y_test[mask]).sum() / mask.sum()
        print(f"  {cat:20s}: {class_acc:.4f} ({class_acc*100:.2f}%)")

## 6. Results Visualization

Visualize the model performance through confusion matrix, per-class accuracy, and feature importance.

In [ ]:
# Classification report
print("\nDetailed Classification Report:")
print("="*80)
report = classification_report(y_test, y_pred, target_names=AUDIO_CATEGORIES, digits=4)
print(report)

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=AUDIO_CATEGORIES,
            yticklabels=AUDIO_CATEGORIES,
            cbar_kws={'label': 'Proportion'})
plt.title('Confusion Matrix (Normalized by True Class)\nExisting Method (64-mel + SAD)', 
          fontsize=14, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/existing_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class accuracy bar chart
per_class_acc = []
for i in range(len(AUDIO_CATEGORIES)):
    mask = y_test == i
    if mask.sum() > 0:
        class_acc = (y_pred[mask] == y_test[mask]).sum() / mask.sum()
        per_class_acc.append(class_acc)
    else:
        per_class_acc.append(0)

plt.figure(figsize=(12, 6))
bars = plt.bar(AUDIO_CATEGORIES, per_class_acc, color='coral', alpha=0.8)
plt.axhline(y=test_acc, color='r', linestyle='--', alpha=0.5, label=f'Overall accuracy: {test_acc:.2%}')
plt.xlabel('Audio Category', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy - Existing Method (64-mel + SAD)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
plt.legend()

# Add value labels on bars
for bar, acc in zip(bars, per_class_acc):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{acc:.2%}',
             ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/existing_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top confusion pairs analysis
print("\nTop 10 Confusion Pairs (most common misclassifications):")
print("="*80)

confusion_pairs = []
for i in range(len(AUDIO_CATEGORIES)):
    for j in range(len(AUDIO_CATEGORIES)):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'True': AUDIO_CATEGORIES[i],
                'Predicted': AUDIO_CATEGORIES[j],
                'Count': cm[i, j],
                'Proportion': cm_normalized[i, j]
            })

df_confusion = pd.DataFrame(confusion_pairs).sort_values('Count', ascending=False)
print(df_confusion.head(10).to_string(index=False))

In [ ]:
# Feature importance (64 features)
feature_importance = clf_final.feature_importances_

plt.figure(figsize=(14, 6))
plt.plot(range(64), feature_importance, linewidth=2, color='coral')
plt.fill_between(range(64), feature_importance, alpha=0.3, color='coral')
plt.xlabel('Feature Index (first 1/3 of 192-D vector)', fontsize=12)
plt.ylabel('Feature Importance', fontsize=12)
plt.title('Feature Importance Across 64 Features\n(Sliced from 192-D: 64 log-mel + 64 delta + 64 delta-delta)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/existing_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTop 10 most important features:")
top_indices = np.argsort(feature_importance)[-10:][::-1]
for idx in top_indices:
    print(f"  Feature {idx:2d}: {feature_importance[idx]:.6f}")

## 7. Model Persistence

Save the trained model, scaler, and evaluation results for future use.

In [ ]:
# Save model
model_path = f'{OUTPUT_DIR}/rf_existing_64mel.pkl'
joblib.dump(clf_final, model_path)
print(f"Model saved to: {model_path}")

# Save scaler
scaler_path = f'{OUTPUT_DIR}/scaler_existing.pkl'
joblib.dump(scaler, scaler_path)
print(f"Scaler saved to: {scaler_path}")

# Save metadata
metadata = {
    'method': 'existing',
    'n_estimators': best_n_estimators,
    'criterion': 'gini',
    'feature_dim': 64,
    'feature_config': FEATURE_CONFIG,
    'test_accuracy': float(test_acc),
    'val_accuracy': float(best_val_acc),
    'seed': SEED,
    'quality_threshold': QUALITY_THRESHOLD,
    'audio_categories': AUDIO_CATEGORIES,
    'train_samples': int(X_train.shape[0]),
    'val_samples': int(X_val.shape[0]),
    'test_samples': int(X_test.shape[0]),
    'scaler_params': {
        'with_mean': True,
        'with_std': False
    }
}

metadata_path = f'{OUTPUT_DIR}/existing_metadata.pkl'
joblib.dump(metadata, metadata_path)
print(f"Metadata saved to: {metadata_path}")

# Save predictions
np.save(f'{OUTPUT_DIR}/y_test_existing.npy', y_test)
np.save(f'{OUTPUT_DIR}/y_pred_existing.npy', y_pred)
np.save(f'{OUTPUT_DIR}/y_proba_existing.npy', y_proba)
print(f"Predictions saved to: {OUTPUT_DIR}/y_test_existing.npy, y_pred_existing.npy, y_proba_existing.npy")

print("\nAll artifacts saved successfully!")

## 8. Inference Example

Demonstrate the complete inference pipeline including preprocessing.

In [ ]:
def predict_audio_category(audio_array, sr=16000, model=None, scaler=None):
    """
    Predict the audio category for a given audio sample (complete pipeline)
    
    Parameters:
    -----------
    audio_array : np.ndarray
        Audio waveform
    sr : int
        Sampling rate
    model : sklearn model
        Trained RandomForest model (if None, loads from disk)
    scaler : sklearn StandardScaler
        Fitted StandardScaler (if None, loads from disk)
    
    Returns:
    --------
    dict : Prediction results
    """
    # Load model and scaler if not provided
    if model is None:
        model = joblib.load(f'{OUTPUT_DIR}/rf_existing_64mel.pkl')
    if scaler is None:
        scaler = joblib.load(f'{OUTPUT_DIR}/scaler_existing.pkl')
    
    # Extract features (includes normalization, SAD, etc.)
    features = extract_existing_features(audio_array, sr)
    
    if features is None:
        return {'error': 'Feature extraction failed (audio too short after SAD or invalid)'}
    
    # Apply scaler
    features_2d = features.reshape(1, -1)
    features_scaled = scaler.transform(features_2d)
    
    # Predict
    pred_label = model.predict(features_scaled)[0]
    pred_proba = model.predict_proba(features_scaled)[0]
    
    # Get top 3 predictions
    top3_indices = np.argsort(pred_proba)[-3:][::-1]
    top3_predictions = [
        {
            'category': AUDIO_CATEGORIES[idx],
            'probability': float(pred_proba[idx])
        }
        for idx in top3_indices
    ]
    
    return {
        'predicted_category': AUDIO_CATEGORIES[pred_label],
        'confidence': float(pred_proba[pred_label]),
        'top3_predictions': top3_predictions
    }

# Test on a sample from test set
print("Testing inference on a sample from test set...\n")

test_example = ds_filtered['test'][0]
test_audio = test_example['audio']['array']
test_sr = test_example['audio']['sampling_rate']
true_category = test_example['audio_type']

result = predict_audio_category(test_audio, test_sr, clf_final, scaler)

if 'error' not in result:
    print(f"True category: {true_category}")
    print(f"Predicted category: {result['predicted_category']}")
    print(f"Confidence: {result['confidence']:.2%}")
    print(f"\nTop 3 predictions:")
    for i, pred in enumerate(result['top3_predictions'], 1):
        print(f"  {i}. {pred['category']:20s} - {pred['probability']:.2%}")
else:
    print(f"Error: {result['error']}")

print("\n" + "="*80)
print("Inference function ready for use!")
print("="*80)

## Summary

This notebook successfully adapted the technical validation pipeline for 9-class sound category classification:

**Results**:
- Test accuracy achieved (see above)
- Expected range: 43-53%
- Heavy preprocessing with SAD and normalization

**Key Features of This Method**:
- Sound Activity Detection removes silence regions
- Amplitude normalization applied
- 64-mel log-spectrogram with delta features
- StandardScaler normalization (mean centering only)
- Mysterious feature slicing (192-D → 64-D)

**Adaptations for 16 kHz**:
- Adjusted window/hop parameters to maintain temporal resolution
- Reduced n_fft and fmax to match Nyquist frequency
- All other processing identical to original pipeline

**Saved Artifacts**:
- `rf_existing_64mel.pkl` - Trained model
- `scaler_existing.pkl` - Fitted StandardScaler
- `existing_metadata.pkl` - Configuration and results
- `y_test_existing.npy`, `y_pred_existing.npy`, `y_proba_existing.npy` - Predictions

**Comparison with Paper Method**:
- Paper method (128-mel, no preprocessing): ~54-58%
- Existing method (64-mel + SAD): ~43-53%
- Difference: ~5-15 percentage points
- Suggests simpler approach works better for this task

**Next Steps**:
- Investigate why heavy preprocessing reduces accuracy
- Try without SAD to see impact
- Experiment with full 192-D features (no slicing)
- Compare feature importance between methods